# Cortex Search: Interactive vs Batch Comparison
Compare 100 and 1000 searches using interactive SDK vs CORTEX_SEARCH_BATCH

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import time
import pandas as pd
from snowflake.snowpark.context import get_active_session
from snowflake.core import Root

# Get active Snowsight session
session = get_active_session()

# Set database context
session.sql("USE DATABASE BATCH_DEMO").collect()
session.sql("USE SCHEMA PUBLIC").collect()

# Connect to Cortex Search service via SDK
root = Root(session)
search_service = root.databases["BATCH_DEMO"].schemas["PUBLIC"].cortex_search_services["WIKIPEDIA_SEARCH_SERVICE"]

SERVICE_NAME = 'BATCH_DEMO.PUBLIC.WIKIPEDIA_SEARCH_SERVICE'

print(f"Connected to: {SERVICE_NAME}")

---
## Test 1: 100 Queries

In [ ]:
# =============================================================================
# GENERATE 100 SEARCH QUERIES
# =============================================================================
NUM_QUERIES_100 = 100

queries_df_100 = session.sql(f"""
    SELECT TITLE AS query_text 
    FROM BATCH_DEMO.PUBLIC.WIKIPEDIA_ARTICLES 
    ORDER BY RANDOM() 
    LIMIT {NUM_QUERIES_100}
""").to_pandas()

search_queries_100 = queries_df_100['QUERY_TEXT'].tolist()
print(f"Generated {len(search_queries_100)} search queries")
print(f"Sample: {search_queries_100[:3]}")

In [ ]:
# =============================================================================
# TEST 1a: INTERACTIVE SEARCH (100 individual SDK calls)
# =============================================================================
print("Running 100 interactive searches...")

interactive_results_100 = []
test_start = time.time()

for i, query in enumerate(search_queries_100):
    try:
        user_start = time.time()
        resp = search_service.search(query=query, columns=["RAW_TEXT"], limit=5)
        _ = resp.to_json()
        user_latency = (time.time() - user_start) * 1000
        interactive_results_100.append({'query_num': i + 1, 'latency_ms': user_latency})
        if (i + 1) % 20 == 0:
            print(f"  Completed {i + 1}/100 queries...")
    except Exception as e:
        print(f"  Error on query {i + 1}: {str(e)[:50]}")

interactive_total_100 = time.time() - test_start
df_interactive_100 = pd.DataFrame(interactive_results_100)

print(f"\nINTERACTIVE (100 queries):")
print(f"  Total time: {interactive_total_100:.1f}s")
print(f"  Avg latency: {df_interactive_100['latency_ms'].mean():.0f}ms")
print(f"  Throughput: {NUM_QUERIES_100 / interactive_total_100:.1f} queries/sec")

In [ ]:
# =============================================================================
# TEST 1b: BATCH SEARCH (100 queries in ONE call)
# =============================================================================
print("Running 100 batch searches in ONE call...")

session.sql("CREATE OR REPLACE TEMPORARY TABLE temp_100_queries (query_text VARCHAR)").collect()
values = ",".join([f"('{q.replace(chr(39), chr(39)+chr(39))}')" for q in search_queries_100])
session.sql(f"INSERT INTO temp_100_queries VALUES {values}").collect()

batch_start = time.time()
batch_sql = f"""
    SELECT q.query_text, s.* 
    FROM temp_100_queries AS q, 
    LATERAL CORTEX_SEARCH_BATCH(
        service_name => '{SERVICE_NAME}', 
        query => q.query_text, 
        limit => 5
    ) AS s
"""
batch_results_100 = session.sql(batch_sql).collect()
batch_total_100 = time.time() - batch_start

print(f"\nBATCH (100 queries):")
print(f"  Total time: {batch_total_100:.1f}s")
print(f"  Results returned: {len(batch_results_100)} rows")
print(f"  Throughput: {NUM_QUERIES_100 / batch_total_100:.1f} queries/sec")

In [ ]:
# =============================================================================
# COMPARISON: 100 QUERIES
# =============================================================================
print("="*70)
print("COMPARISON: Interactive vs Batch (100 queries)")
print("="*70)

speedup_100 = interactive_total_100 / batch_total_100 if batch_total_100 > 0 else 0

print(f"  Interactive: {interactive_total_100:.1f}s | {NUM_QUERIES_100 / interactive_total_100:.1f} qps")
print(f"  Batch:       {batch_total_100:.1f}s | {NUM_QUERIES_100 / batch_total_100:.1f} qps")
print(f"\n  Winner: {'BATCH' if speedup_100 > 1 else 'INTERACTIVE'} ({speedup_100:.1f}x)")

---
## Test 2: 1000 Queries

In [ ]:
# =============================================================================
# GENERATE 1000 SEARCH QUERIES
# =============================================================================
NUM_QUERIES_1000 = 1000

queries_df_1000 = session.sql(f"""
    SELECT TITLE AS query_text 
    FROM BATCH_DEMO.PUBLIC.WIKIPEDIA_ARTICLES 
    ORDER BY RANDOM() 
    LIMIT {NUM_QUERIES_1000}
""").to_pandas()

search_queries_1000 = queries_df_1000['QUERY_TEXT'].tolist()
print(f"Generated {len(search_queries_1000)} search queries")
print(f"Sample: {search_queries_1000[:3]}")

In [ ]:
# =============================================================================
# TEST 2a: INTERACTIVE SEARCH (1000 individual SDK calls)
# =============================================================================
print("Running 1000 interactive searches...")
print("="*60)

interactive_results_1000 = []
test_start = time.time()

for i, query in enumerate(search_queries_1000):
    try:
        user_start = time.time()
        resp = search_service.search(query=query, columns=["RAW_TEXT"], limit=5)
        _ = resp.to_json()
        user_latency = (time.time() - user_start) * 1000
        interactive_results_1000.append({'query_num': i + 1, 'latency_ms': user_latency})
        if (i + 1) % 200 == 0:
            print(f"  Completed {i + 1}/1000 queries...")
    except Exception as e:
        print(f"  Error on query {i + 1}: {str(e)[:50]}")

interactive_total_1000 = time.time() - test_start
df_interactive_1000 = pd.DataFrame(interactive_results_1000)

print(f"\nINTERACTIVE (1000 queries):")
print(f"  Total time: {interactive_total_1000:.1f}s")
print(f"  Avg latency: {df_interactive_1000['latency_ms'].mean():.0f}ms")
print(f"  Throughput: {NUM_QUERIES_1000 / interactive_total_1000:.1f} queries/sec")

In [ ]:
# =============================================================================
# TEST 2b: BATCH SEARCH (1000 queries in ONE call)
# =============================================================================
print("Running 1000 batch searches in ONE call...")
print("="*60)

session.sql("CREATE OR REPLACE TEMPORARY TABLE temp_1000_queries (query_text VARCHAR)").collect()

# Insert in chunks to avoid SQL size limits
chunk_size = 500
for i in range(0, len(search_queries_1000), chunk_size):
    chunk = search_queries_1000[i:i+chunk_size]
    values = ",".join([f"('{q.replace(chr(39), chr(39)+chr(39))}')" for q in chunk])
    session.sql(f"INSERT INTO temp_1000_queries VALUES {values}").collect()

batch_start = time.time()
batch_sql = f"SELECT q.query_text, s.* FROM temp_1000_queries AS q, LATERAL CORTEX_SEARCH_BATCH(service_name => '{SERVICE_NAME}', query => q.query_text, limit => 5) AS s"
batch_results_1000 = session.sql(batch_sql).collect()
batch_total_1000 = time.time() - batch_start

print(f"\nBATCH (1000 queries):")
print(f"  Total time: {batch_total_1000:.1f}s")
print(f"  Results returned: {len(batch_results_1000)} rows")
print(f"  Throughput: {NUM_QUERIES_1000 / batch_total_1000:.1f} queries/sec")

In [ ]:
# =============================================================================
# COMPARISON: 1000 QUERIES
# =============================================================================
print("="*70)
print("COMPARISON: Interactive vs Batch (1000 queries)")
print("="*70)

speedup_1000 = interactive_total_1000 / batch_total_1000 if batch_total_1000 > 0 else 0

print(f"  Interactive: {interactive_total_1000:.1f}s | {NUM_QUERIES_1000 / interactive_total_1000:.1f} qps")
print(f"  Batch:       {batch_total_1000:.1f}s | {NUM_QUERIES_1000 / batch_total_1000:.1f} qps")
print(f"\n  Winner: {'BATCH' if speedup_1000 > 1 else 'INTERACTIVE'} ({speedup_1000:.1f}x)")

---
## Test 3: 10000 Queries

In [ ]:
NUM_QUERIES_10000 = 10000

print("Generating 10,000 search queries...")
queries_df_10000 = session.sql(f"""
    SELECT TITLE AS query_text 
    FROM BATCH_DEMO.PUBLIC.WIKIPEDIA_ARTICLES 
    ORDER BY RANDOM() 
    LIMIT {NUM_QUERIES_10000}
""").to_pandas()

search_queries_10000 = queries_df_10000['QUERY_TEXT'].tolist()
print(f"Generated {len(search_queries_10000)} search queries")


In [ ]:
#This test truly demonstrates the power of batch search at scale. 
#Running 10,000 interactive searches would take nearly an hour, so we only run the batch version and make an assumption that interactive would scale linearly

In [ ]:
print("Running 10,000 batch searches in ONE call...")

session.sql("CREATE OR REPLACE TEMPORARY TABLE temp_10000_queries (query_text VARCHAR)").collect()

# Insert in chunks to avoid SQL size limits
chunk_size = 500
for i in range(0, len(search_queries_10000), chunk_size):
    chunk = search_queries_10000[i:i+chunk_size]
    values = ",".join([f"('{q.replace(chr(39), chr(39)+chr(39))}')" for q in chunk])
    session.sql(f"INSERT INTO temp_10000_queries VALUES {values}").collect()
    if (i + chunk_size) % 2000 == 0:
        print(f"  Inserted {min(i + chunk_size, len(search_queries_10000))}/{len(search_queries_10000)} queries...")

batch_start = time.time()
batch_sql = f"""
    SELECT q.query_text, s.* 
    FROM temp_10000_queries AS q, 
    LATERAL CORTEX_SEARCH_BATCH(
        service_name => '{SERVICE_NAME}', 
        query => q.query_text, 
        limit => 5
    ) AS s
"""
batch_results_10000 = session.sql(batch_sql).collect()
batch_total_10000 = time.time() - batch_start

print(f"\nBATCH (10,000 queries):")
print(f"  Total time: {batch_total_10000:.1f}s")
print(f"  Results returned: {len(batch_results_10000)} rows")
print(f"  Throughput: {NUM_QUERIES_10000 / batch_total_10000:.1f} queries/sec")

# Estimate how long interactive would take
estimated_interactive = NUM_QUERIES_10000 * (df_interactive_1000['latency_ms'].mean() / 1000)
print(f"\n  Estimated interactive time: {estimated_interactive:.0f}s ({estimated_interactive/60:.1f} minutes)")
print(f"  Batch speedup: ~{estimated_interactive / batch_total_10000:.0f}x faster!")

---
## Final Summary

In [ ]:
# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("="*70)
print("FINAL SUMMARY: Interactive vs Batch")
print("="*70)

speedup_100 = interactive_total_100 / batch_total_100 if batch_total_100 > 0 else 0
speedup_1000 = interactive_total_1000 / batch_total_1000 if batch_total_1000 > 0 else 0

summary = pd.DataFrame({
    'Queries': [100, 1000, 10000],
    'Interactive (s)': [
        f"{interactive_total_100:.1f}", 
        f"{interactive_total_1000:.1f}", 
        f"~{estimated_interactive:.0f} (est)"
    ],
    'Batch (s)': [
        f"{batch_total_100:.1f}", 
        f"{batch_total_1000:.1f}", 
        f"{batch_total_10000:.1f}"
    ],
    'Speedup': [f"{speedup_100:.1f}x", f"{speedup_1000:.1f}x", f"~{estimated_interactive/batch_total_10000:.0f}x"]
})

print(summary.to_string(index=False))

print("\n" + "="*70)
print("RECOMMENDATION:")
print("="*70)
print("  - For real-time, user-facing search: Use Interactive SDK")
print("  - For batch processing (100+ queries): Use CORTEX_SEARCH_BATCH")
print("  - Batch has a small bootstrap cost - amortize it with larger batches!")